# 🍏 Apple Generative Imagery Systems - Studio v2
### 3D Spatial LookDev Pipeline (Direct PNG & OpenEXR + Custom LoRA)
---
이 주피터 노트북은 Houdini의 **3D Depth, Normal, Id (PNG 및 32-bit EXR 원본)**을 ComfyUI에서 실시간으로 직접 읽어들여 최고급 애플 룩 화보를 렌더링하는 올인원 스튜디오입니다.

In [ ]:
# 1. GPU 사양 확인
!nvidia-smi

In [ ]:
# 2. Google Drive 마운트 (내 LoRA 모델 및 3D 가이드 연동)
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

In [ ]:
# 3. ComfyUI 최신 엔진 + OpenEXR/PNG 정밀 처리 노드 자동 설치
%cd /content
!git clone https://github.com/comfyanonymous/ComfyUI.git 2>/dev/null || true
%cd /content/ComfyUI
!pip install -q -r requirements.txt
!pip install -q opencv-python-headless openexr

# 📦 [핵심] 3D Depth(0.2~1.0 맵핑) + Normal + Color ID (PNG/EXR) 직접 로더 커스텀 노드 생성
import os
os.makedirs("/content/ComfyUI/custom_nodes/my_exr_loader", exist_ok=True)
with open("/content/ComfyUI/custom_nodes/my_exr_loader/load_exr.py", "w", encoding="utf-8") as f:
    f.write('''import os, glob, torch, numpy as np
os.environ["OPENCV_IO_ENABLE_OPENEXR"] = "1"
import cv2

class LoadNativeEXR:
    @classmethod
    def INPUT_TYPES(s):
        return {
            "required": {
                "folder_path": ("STRING", {"default": "/content/drive/MyDrive/Generative-Imagery-Systems/apple_lora_project/3d_guides/0011", "multiline": False}),
                "exr_file_name": ("STRING", {"default": "Depth.png", "multiline": False}),
                "pass_type": (["Depth", "Normal", "Color_ID", "Direct_RGB"], {"default": "Depth"}),
            }
        }

    RETURN_TYPES = ("IMAGE",)
    RETURN_NAMES = ("IMAGE",)
    FUNCTION = "load_and_process_exr"
    CATEGORY = "3D_VFX_Pipeline"

    def load_and_process_exr(self, folder_path, exr_file_name, pass_type):
        target_dir = folder_path.strip()
        if not os.path.exists(target_dir):
            base_search = os.path.basename(target_dir) if os.path.basename(target_dir) else "0011"
            found_folders = glob.glob(f"/content/drive/MyDrive/**/{base_search}", recursive=True)
            if found_folders:
                target_dir = found_folders[0]
                print(f"🔍 [LoadNativeEXR] 자동 추적된 폴더: {target_dir}")
            else:
                raise FileNotFoundError(f"❌ 폴더를 찾을 수 없습니다: {folder_path}")

        target_file = exr_file_name.strip()
        full_path = os.path.join(target_dir, target_file)
        if not os.path.exists(full_path):
            all_files = [f for f in os.listdir(target_dir) if f.endswith('.png') or f.endswith('.exr') or f.endswith('.jpg')]
            matched = [f for f in all_files if pass_type.lower() in f.lower() or os.path.splitext(target_file)[0].lower() in f.lower()]
            if matched:
                full_path = os.path.join(target_dir, matched[0])
                print(f"🎯 [LoadNativeEXR] 자동 매칭된 파일: {matched[0]}")
            elif all_files:
                full_path = os.path.join(target_dir, all_files[0])
            else:
                raise FileNotFoundError(f"❌ {target_dir} 안에 {pass_type} 파일(PNG/EXR)이 없습니다.")

        print(f"📦 [LoadNativeEXR] 3D 파일 로딩 중: {full_path} ({pass_type})")
        img = cv2.imread(full_path, cv2.IMREAD_UNCHANGED)
        if img is None:
            raise ValueError(f"❌ 파일 읽기 실패: {full_path}")

        is_png = full_path.lower().endswith('.png') or full_path.lower().endswith('.jpg')

        # 1) Depth 정밀 매핑 (0.2 ~ 1.0 -> 1.0 ~ 0.0)
        if pass_type == "Depth":
            d_raw = (img[:, :, 0] if len(img.shape) == 3 else img).astype(np.float32)
            if is_png or img.dtype == np.uint8:
                d_raw = d_raw / 255.0

            d_min = float(np.min(d_raw[d_raw > 0.01])) if np.any(d_raw > 0.01) else 0.2
            d_max = float(np.max(d_raw)) if np.any(d_raw > 0.01) else 1.0
            
            if d_max > d_min:
                d_norm = np.clip((d_raw - d_min) / (d_max - d_min), 0.0, 1.0)
                d_final = 1.0 - d_norm
            else:
                d_final = np.zeros_like(d_raw)

            d_final[d_raw < 0.001] = 0.0
            rgb = np.stack([d_final, d_final, d_final], axis=-1)

        # 2) Normal BGR->RGB 및 디코딩
        elif pass_type == "Normal":
            if is_png or img.dtype == np.uint8:
                rgb = img.astype(np.float32) / 255.0
            else:
                rgb = np.clip((img + 1.0) * 0.5, 0.0, 1.0)
            if len(rgb.shape) == 3 and rgb.shape[-1] >= 3:
                rgb = rgb[:, :, [2, 1, 0]]

        # 3) Color ID (PNG sRGB 그대로 VAE 주입)
        else:
            if is_png or img.dtype == np.uint8:
                rgb = img.astype(np.float32) / 255.0
                if len(rgb.shape) == 3 and rgb.shape[-1] >= 3:
                    rgb = rgb[:, :, [2, 1, 0]]
            else:
                rgb = np.clip(img, 0.0, 1.0)
                if len(rgb.shape) == 3 and rgb.shape[-1] >= 3:
                    rgb = rgb[:, :, [2, 1, 0]]

            rgb = np.clip(rgb, 0.0, 1.0)

        # 🛡️ 3채널 강제 고정
        if len(rgb.shape) == 2:
            rgb = np.stack([rgb, rgb, rgb], axis=-1)
        elif len(rgb.shape) == 3:
            if rgb.shape[-1] == 1:
                rgb = np.concatenate([rgb, rgb, rgb], axis=-1)
            elif rgb.shape[-1] > 3:
                rgb = rgb[:, :, :3]

        # 🛡️ 해상도를 16:9 정밀 1024x576으로 고정
        h, w = rgb.shape[:2]
        if w != 1024 or h != 576:
            interp = cv2.INTER_NEAREST if pass_type == "Color_ID" else cv2.INTER_AREA
            rgb = cv2.resize(rgb, (1024, 576), interpolation=interp)
        tensor = torch.from_numpy(np.ascontiguousarray(rgb, dtype=np.float32)).unsqueeze(0)
        return (tensor,)

NODE_CLASS_MAPPINGS = {"LoadNativeEXR": LoadNativeEXR}
NODE_DISPLAY_NAME_MAPPINGS = {"LoadNativeEXR": "📦 Load 3D Pass (PNG & EXR Native)"}
''')

print('✅ ComfyUI 엔진 및 완벽 방어형 PNG/EXR 노드 장착 완료!')

In [ ]:
# 4. SDXL Base + VAE + ControlNet (Depth & Normal) + LoRA 가중치 연동
import glob, os
print('🚀 필수 AI 모델 가중치 초고속 다운로드 중...')
# 1) SDXL Base 1.0 (6.4GB)
!wget -c https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors -P /content/ComfyUI/models/checkpoints/

# 2) SDXL VAE Fix (335MB)
!wget -c https://huggingface.co/madebyollin/sdxl-vae-fp16-fix/resolve/main/sdxl_vae.safetensors -P /content/ComfyUI/models/vae/

# 3) SDXL ControlNet Depth (2.5GB)
!wget -c https://huggingface.co/diffusers/controlnet-depth-sdxl-1.0/resolve/main/diffusion_pytorch_model.safetensors -O /content/ComfyUI/models/controlnet/controlnet-depth-sdxl-1.0.safetensors

# 4) SDXL ControlNet Normal / Union (2.3GB)
!wget -c https://huggingface.co/xinsir/controlnet-union-sdxl-1.0/resolve/main/diffusion_pytorch_model.safetensors -O /content/ComfyUI/models/controlnet/controlnet-normal-sdxl-1.0.safetensors

# 5) LoRA 가중치 자동 연동
lora_target_dir = "/content/ComfyUI/models/loras"
os.makedirs(lora_target_dir, exist_ok=True)
lora_found = glob.glob("/content/drive/MyDrive/**/apple_minimal_craft_sdxl_v1.safetensors", recursive=True)
if lora_found:
    src_lora = lora_found[0]
    dst_lora = os.path.join(lora_target_dir, "apple_minimal_craft_sdxl_v1.safetensors")
    if not os.path.exists(dst_lora):
        os.symlink(src_lora, dst_lora)
    print(f'✅ LoRA 연동 완료: {src_lora}')

print('✅ 모든 모델 준비 완료!')

In [ ]:
# 5. 🌐 403 에러 완벽 해결 Cloudflare 직통 터널 가동
!wget -q -c -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
!pkill -f "main.py" 2>/dev/null || true
!pkill -f "cloudflared" 2>/dev/null || true

import threading, time

def run_comfy():
    !python /content/ComfyUI/main.py --listen 0.0.0.0 --port 8188 --enable-cors-header "*" --highvram --dont-upcast-attention

threading.Thread(target=run_comfy, daemon=True).start()
time.sleep(4)

# 403 차단을 뚫는 --http-host-header 옵션 장착
!cloudflared tunnel --url http://127.0.0.1:8188 --http-host-header=localhost:8188
